<a href="https://colab.research.google.com/github/naubergois/curso-transfer-colab/blob/main/exercicios/aula1-exercicio4-o-que-ainda-treina.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# Aula 1, exercício 4: O que ainda treina

Depois de congelar a base e trocar a cabeça, quantos parâmetros de `modelo` ainda podem mudar?

O exercício é da aula [1. O peso que viajou](https://colab.research.google.com/github/naubergois/curso-transfer-colab/blob/main/notebooks/01-o-peso-que-viajou.ipynb), e este notebook roda sozinho. Use **Ambiente de execução → Executar tudo**: ele prepara o modelo, deixa uma célula para você tentar e, no fim, mostra uma solução já conferida.

In [ ]:
!pip install -q torch torchvision

## Preparação

A preparação carrega a ResNet-18 com os pesos do ImageNet, guarda a rede intacta e monta uma cópia com a base congelada e a cabeça nova de duas saídas.

In [ ]:
def contar(modelo, so_treinavel=False):
    total = 0
    for p in modelo.parameters():
        if so_treinavel and not p.requires_grad:
            continue
        total += p.numel()
    return total


import torch
from torch import nn
from torchvision.models import ResNet18_Weights, resnet18

torch.manual_seed(7)

rede = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
rede.eval()
entrada = rede.fc.in_features
saida_antiga = rede.fc.out_features

modelo = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
modelo.eval()
for p in modelo.parameters():
    p.requires_grad = False
modelo.fc = nn.Linear(entrada, 2)

print(f"pesos que vieram: {contar(rede)}")
print(f"cabeça antiga: {entrada} -> {saida_antiga}")
print(f"cabeça nova: {modelo.fc.in_features} -> {modelo.fc.out_features}")
print(f"ainda treinam: {contar(modelo, so_treinavel=True)}")
print(f"parados: {contar(modelo) - contar(modelo, so_treinavel=True)}")
print(f"conv1 pode mudar? {modelo.conv1.weight.requires_grad}")

In [ ]:
# @title Correção automática (rode, não precisa ler)
VERIFICAR = {
    1: lambda r: int(r) == contar(rede),
    2: lambda r: int(r) == saida_antiga,
    3: lambda r: int(r) == entrada,
    4: lambda r: int(r) == contar(modelo, so_treinavel=True),
    5: lambda r: int(r) == contar(modelo) - contar(modelo, so_treinavel=True),
    6: lambda r: bool(r) is False,
}

_acertos = {}


def conferir(n, resposta):
    if resposta is None:
        print(f"Exercício {n}: ainda sem resposta.")
        return
    try:
        certo = bool(VERIFICAR[n](resposta))
    except Exception as erro:
        certo = False
        print(f"  (a verificação deu erro: {erro})")
    _acertos[n] = certo
    print(f"Exercício {n}: " + ("✅ certo!" if certo else "❌ ainda não. Releia o enunciado e tente de novo."))


def placar():
    print(f"{sum(_acertos.values())} de {len(VERIFICAR)} exercícios certos.")

## Sua vez

Depois de congelar a base e trocar a cabeça, quantos parâmetros de `modelo` ainda podem mudar?

*Dica:* contar(modelo, so_treinavel=True) ignora quem está com requires_grad False.

Troque o `None` pela sua resposta e rode a célula.

In [ ]:
resposta = None

conferir(4, resposta)

## Uma solução

Compare com o que você fez. Existem outros caminhos certos: o que vale é a `conferir` aprovar.

In [ ]:
resposta = contar(modelo, so_treinavel=True)

conferir(4, resposta)

## O que fica

Próximo: [Aula 1, exercício 5: O que ficou parado](https://colab.research.google.com/github/naubergois/curso-transfer-colab/blob/main/exercicios/aula1-exercicio5-o-que-ficou-parado.ipynb).